In [1]:
# Download the dataset 
# Dataset URL : https://www.kaggle.com/datasets/ayushmandatta1/deepdetect-2025
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ayushmandatta1/deepdetect-2025")

print("Path to dataset files:", path)
root_folder = "/kaggle/input/deepdetect-2025/ddata"

Path to dataset files: /kaggle/input/datasets/ayushmandatta1/deepdetect-2025


In [2]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# 1. Load MobileNetV2 sebagai base
base_model = MobileNetV2(input_shape=(224, 224, 3),
                         include_top=False, 
                         weights='imagenet')

# 2. Freeze base model
base_model.trainable = False

# 3. Rakit model baru
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2), # Mencegah overfitting
    layers.Dense(1, activation='sigmoid') # Output: 0 atau 1
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

2026-06-02 05:01:08.226201: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780376468.447656      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780376468.507259      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780376468.973881      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780376468.973923      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780376468.973926      23 computation_placer.cc:177] computation placer alr

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Menyiapkan "pelatih" untuk data latih (Train) dengan augmentasi
train_datagen = ImageDataGenerator(
    rescale=1./255,          # Normalisasi: mengubah piksel 0-255 menjadi 0-1
    rotation_range=20,       # Variasi: memutar gambar hingga 20 derajat
    horizontal_flip=True,    # Variasi: membalik gambar kiri-kanan
    width_shift_range=0.1,   # Variasi: menggeser gambar sedikit ke kiri/kanan
    height_shift_range=0.1   # Variasi: menggeser gambar sedikit ke atas/bawah
)

# 2. Menyiapkan pemroses untuk data uji (Test)
test_datagen = ImageDataGenerator(
    rescale=1./255           # HANYA normalisasi
)

In [4]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Memuat MobileNetV2 dengan bobot pretrained dari ImageNet
base_model = MobileNetV2(input_shape=(224, 224, 3),
                         include_top=False, 
                         weights='imagenet')

# Membekukan base_model agar fitur dasar yang sudah dipelajari tidak rusak
base_model.trainable = False

# Menyusun model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2), # Menghindari 'menghafal' berlebihan (overfitting)
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [5]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# --- 1. DEFINISI PATH ---
# Menggunakan path yang sudah kita temukan tadi
train_path = '/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train'
test_path = '/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/test'

# --- 2. DATA AUGMENTATION ---
# Menyiapkan generator untuk melatih model dengan variasi gambar
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

# --- 3. MEMBANGUN MODEL (TAHAP 1: FROZEN) ---
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # Mengunci MobileNetV2

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("\n--- Memulai Tahap 1: Latihan Kepala Model (Warm-up) ---")
model.fit(train_generator, epochs=5, validation_data=test_generator)

# --- 4. FINE-TUNING (TAHAP 2: UNFREEZE PARTIAL) ---
print("\n--- Memulai Tahap 2: Fine-tuning untuk Akurasi Tinggi ---")
base_model.trainable = True

# Kita hanya buka 50 lapisan terakhir agar tidak merusak fitur dasar
for layer in base_model.layers[:100]:
    layer.trainable = False

# Gunakan Learning Rate yang SANGAT kecil
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

history = model.fit(train_generator, epochs=10, validation_data=test_generator)

Found 90409 images belonging to 2 classes.
Found 21776 images belonging to 2 classes.

--- Memulai Tahap 1: Latihan Kepala Model (Warm-up) ---
Epoch 1/5


I0000 00:00:1780376600.187844      65 service.cc:152] XLA service 0x7a96980026e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780376600.187899      65 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1780376601.364348      65 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1780376607.463958      65 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1929s 678ms/step - accuracy: 0.6860 - loss: 0.5914 - val_accuracy: 0.7773 - val_loss: 0.4774
Epoch 2/5
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1215s 430ms/step - accuracy: 0.7058 - loss: 0.5690 - val_accuracy: 0.7172 - val_loss: 0.5457
Epoch 3/5
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1231s 436ms/step - accuracy: 0.7053 - loss: 0.5697 - val_accuracy: 0.7846 - val_loss: 0.4690
Epoch 4/5
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1215s 430ms/step - accuracy: 0.7047 - loss: 0.5721 - val_accuracy: 0.6984 - val_loss: 0.5612
Epoch 5/5
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1231s 436ms/step - accuracy: 0.7051 - loss: 0.5694 - val_accuracy: 0.7337 - val_loss: 0.5295

--- Memulai Tahap 2: Fine-tuning untuk Akurasi Tinggi ---
Epoch 1/10


2026-06-02 06:57:12.394470: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-02 06:57:12.591105: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


 357/2826 ━━━━━━━━━━━━━━━━━━━━ 15:45 383ms/step - accuracy: 0.6126 - loss: 0.8441

2026-06-02 06:59:38.642293: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-02 06:59:38.837549: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1255s 437ms/step - accuracy: 0.7504 - loss: 0.5185 - val_accuracy: 0.6908 - val_loss: 0.6618
Epoch 2/10
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1190s 421ms/step - accuracy: 0.8391 - loss: 0.3619 - val_accuracy: 0.6557 - val_loss: 0.7860
Epoch 3/10
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1192s 422ms/step - accuracy: 0.8731 - loss: 0.2949 - val_accuracy: 0.6943 - val_loss: 0.7213
Epoch 4/10
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1174s 416ms/step - accuracy: 0.8941 - loss: 0.2543 - val_accuracy: 0.6859 - val_loss: 0.7765
Epoch 5/10
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1247s 441ms/step - accuracy: 0.9064 - loss: 0.2263 - val_accuracy: 0.7009 - val_loss: 0.7333
Epoch 6/10
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1197s 424ms/step - accuracy: 0.9165 - loss: 0.2024 - val_accuracy: 0.7224 - val_loss: 0.6969
Epoch 7/10
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1203s 426ms/step - accuracy: 0.9259 - loss: 0.1846 - val_accuracy: 0.6971 - val_loss: 0.8186
Epoch 8/10
2826/2826 ━━━━━━━━━━━━━━━━━━━━ 1164s 412ms/step - accur